<a href="https://colab.research.google.com/github/janithcyapa/Statistical-Learning-e20452/blob/main/Assignments/Assignment_05_Gaussian_Process_Regression.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **Assignment : Gaussian Process Regression & Linear Regression**
### ME526 : Introduction to Statistical Learning
---
> YAPA W.S.P.Y.J.C.
>
> E/20/452
---

In [2]:
!pip install git+https://github.com/janithcyapa/Statistical-Learning-e20452.git

  Cloning https://github.com/janithcyapa/Statistical-Learning-e20452.git to /tmp/pip-req-build-tcw34sav
  Running command git clone --filter=blob:none --quiet https://github.com/janithcyapa/Statistical-Learning-e20452.git /tmp/pip-req-build-tcw34sav
  Resolved https://github.com/janithcyapa/Statistical-Learning-e20452.git to commit f56b6f8c6a08c109be3e262c832490320e951e19
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  Created wheel for data_wrangling: filename=data_wrangling-0.4.6-py3-none-any.whl size=20150 sha256=c7f9ab46e373a2967724ae81261eb8e4ca7716d24c6ba3d4f2ad37c22e5e2b04
  Stored in directory: /tmp/pip-ephem-wheel-cache-wl02j9lo/wheels/e1/3a/aa/1f58ac076c2d0bbe3f06486762ffa74c6111114aab57df58a1
Successfully built data_wrangling


## **Gaussian Process Regression**

Modeling both the Heating Load (Y1) and Cooling Load (Y2) as a single parameter Gaussian Process need to handle two dependent thermal loads simultaneously, we therefore General Multivariate Formulation is used here.

Instead of two independent scalar processes, define  observation $Y_g$ at any building configuration g as a vector in $\mathbb{R}^2$,
$$Y_g =X_g + ν_g$$

where $X_g ∈ R_2$ is the latent clean thermal load, and $ν_g ∼ N(0,Σν​)$ is the noise vector.
\
\
If we have $n$ building observations, our stacked observation vector $\mathscr{Y}_n$ is in $\mathbb{R}^{2n}$. The joint marginal distribution becomes:


$$\mathscr{Y}_n \sim \mathscr{N} \left( \mu_{Y,n}, K_n + I_n \otimes \Sigma_\nu \right)$$

Here, the covariance matrix $K_n$ is built from a matrix-valued kernel $\kappa(g, g') \in \mathbb{R}^{2 \times 2}$. The diagonal blocks represent the variance of heating and cooling loads independently, while the off-diagonal elements capture the cross-correlation between them.

To estimate the thermal loads $X_g$ for a new building configuration, the predictive mean relies on the block matrix inversion,

$$\mathbb{E}[X_g \mid \mathscr{Y}_n=y_n] = \mu_g + K_{g,n} \left( K_n + I_n \otimes \Sigma_\nu \right)^{-1} \left( y_n - \mu_{Y,n} \right)$$

By modeling them together, the GP leverages the shared architectural features (X1 to X8) and the intrinsic thermodynamic coupling between heating and cooling to create a more robust predictive distribution.

In [15]:
import os
import kagglehub
import pandas as pd
import numpy as np
import plotly.graph_objects as go

from data_wrangling.core import DataInspector
from data_wrangling import PlottingMethods

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.gaussian_process import GaussianProcessRegressor
from sklearn.gaussian_process.kernels import Matern, WhiteKernel, ConstantKernel as C

In [8]:

# Download latest version
kagglepath="elikplim/eergy-efficiency-dataset"
path = kagglehub.dataset_download(kagglepath)

print("Path to dataset files:", path)
print(f"Listing contents of: {path}")
!ls {path}
df=pd.read_csv(path+"/ENB2012_data.csv")
# df

Using Colab cache for faster access to the 'eergy-efficiency-dataset' dataset.
Path to dataset files: /kaggle/input/eergy-efficiency-dataset
Listing contents of: /kaggle/input/eergy-efficiency-dataset
ENB2012_data.csv


In [11]:
# The dataset features X1 - X8
# Responses Y1: Heating Load, Y2: Cooling Load
feature_cols = ['X1', 'X2', 'X3', 'X4', 'X5', 'X6', 'X7', 'X8']
target_cols = ['Y1', 'Y2']

X = df[feature_cols].values
Y = df[target_cols].values

X_train, X_test, Y_train, Y_test = train_test_split(X, Y, test_size=0.2, random_state=42)

scaler_X = StandardScaler()
X_train_scaled = scaler_X.fit_transform(X_train)
X_test_scaled = scaler_X.transform(X_test)


In [12]:
# Gaussian Process Modeling
kernel = C(1.0, (1e-3, 1e3)) * Matern(length_scale=np.ones(8), nu=1.5) + WhiteKernel(noise_level=1, noise_level_bounds=(1e-5, 1e1))

gp = GaussianProcessRegressor(kernel=kernel, n_restarts_optimizer=10, normalize_y=True, random_state=42)
gp.fit(X_train_scaled, Y_train)
Y_pred, Y_std = gp.predict(X_test_scaled, return_std=True)
r2_joint = gp.score(X_test_scaled, Y_test)
print(f"Optimized Kernel: {gp.kernel_}")
print(f"Joint R^2 Score on Test Set: {r2_joint:.4f}")


Optimized Kernel: 25.8**2 * Matern(length_scale=[20.1, 418, 17.4, 745, 1.02e+04, 48.2, 359, 29.3], nu=1.5) + WhiteKernel(noise_level=0.00118)
Joint R^2 Score on Test Set: 0.9969


In [16]:
# Plot Reslusts
num_samples = 40
x_idx = np.arange(num_samples)
y_true = Y_test[:num_samples, 0]
y_mean = Y_pred[:num_samples, 0]
y_std_val = Y_std[:num_samples]

fig = go.Figure()

# 95% Confidence Interval
fig.add_trace(go.Scatter(
    x=np.concatenate([x_idx, x_idx[::-1]]),
    y=np.concatenate([y_mean - 1.96 * y_std_val, (y_mean + 1.96 * y_std_val)[::-1]]),
    fill='toself',
    fillcolor='rgba(0, 255, 200, 0.15)',
    line_color='rgba(255,255,255,0)',
    name='95% Confidence Interval',
    showlegend=True,
    hoverinfo='skip'
))

# Predictive Mean (The Estimate)
fig.add_trace(go.Scatter(
    x=x_idx,
    y=y_mean,
    mode='lines+markers',
    line=dict(color='rgb(0, 255, 200)', width=2.5),
    marker=dict(size=6, color='rgb(0, 255, 200)'),
    name='GP Predictive Mean'
))

# the actual observations
fig.add_trace(go.Scatter(
    x=x_idx,
    y=y_true,
    mode='markers',
    marker=dict(color='white', size=8, line=dict(color='black', width=1)),
    name='Actual Heating Load (Y1)'
))

fig.update_layout(
    title="GPR Predictions vs Actual: Heating Load (First 40 Test Samples)",
    xaxis_title="Test Sample Index",
    yaxis_title="Heating Load (kWh/m²)",
    template="plotly_dark",
    hovermode="x unified",
    plot_bgcolor='rgb(17,17,17)',
    paper_bgcolor='rgb(17,17,17)',
    legend=dict(
        yanchor="top",
        y=0.99,
        xanchor="left",
        x=0.01,
        bgcolor="rgba(0,0,0,0.5)"
    )
)

fig.show()

ValueError: operands could not be broadcast together with shapes (40,) (40,2) 